# MCDI505 · Semana 3 — Modelos avanzados (ARFIMA / GARCH)
**Estudiante:** _Nombre Apellido_ · **Archivo:** `mcdi505_s3inicialapellido.ipynb`

Este es el único notebook de la semana. El **miércoles** entregue la Hoja de decisiones completa (Formativa 3); el **domingo** entregue este mismo archivo terminado, con la hoja corregida según la retroalimentación (Sumativa 3).

**Pregunta de la semana:** ¿la memoria larga o la volatilidad condicional aportan algo al SARIMA en esta serie? Una respuesta negativa bien fundamentada es tan válida como una positiva.

## Hoja de decisiones

| Criterio | Decisión o conclusión | Evidencia clave (cifra, prueba o figura) | Alternativa descartada | Consecuencia |
|---|---|---|---|---|
| 1. Memoria larga | | | | |
| 2. Volatilidad | | | | |
| 3. Contraste con el modelo clásico | | | | |
| 4. Selección del modelo | | | | |

## 0. Configuración, datos y punto de partida

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
warnings.filterwarnings("ignore")
SEMILLA = 505
np.random.seed(SEMILLA)

In [ ]:
# ==============================================================================
# Funciones de apoyo (las mismas de la Sesión 3, más tres envoltorios para DEL)
# No necesita modificarlas.
# ==============================================================================

def calcular_exponente_hurst(serie, min_ventana=8, max_ventana=None, n_ventanas=10):
    """Exponente de Hurst por Rango Reescalado (R/S). Ver Sesión 3, §2.3."""
    serie = np.asarray(serie, dtype=float)
    n = len(serie)
    if max_ventana is None:
        max_ventana = n // 4
    ventanas = np.unique(np.geomspace(min_ventana, max_ventana, num=n_ventanas, dtype=int))
    rs_promedio, ventanas_validas = [], []
    for w in ventanas:
        m = n // w
        if m < 1:
            continue
        rs_bloques = []
        for i in range(m):
            bloque = serie[i * w:(i + 1) * w]
            recorrido = np.cumsum(bloque - np.mean(bloque))
            desviacion = np.std(bloque)
            if desviacion > 0:
                rs_bloques.append((np.max(recorrido) - np.min(recorrido)) / desviacion)
        if rs_bloques:
            rs_promedio.append(np.mean(rs_bloques))
            ventanas_validas.append(w)
    pendiente, _ = np.polyfit(np.log(ventanas_validas), np.log(rs_promedio), 1)
    return pendiente


def aplicar_filtro_fraccional(serie, d):
    """Operador (1-B)^d. d > 0 diferencia fraccionalmente; d < 0 integra. Ver Sesión 3, §2.3."""
    serie = np.asarray(serie, dtype=float)
    n = len(serie)
    pesos = np.zeros(n)
    pesos[0] = 1.0
    for k in range(1, n):
        pesos[k] = pesos[k - 1] * (k - 1 - d) / k
    return np.array([np.sum(pesos[:t + 1] * serie[t::-1]) for t in range(n)])


def garch_log_verosimilitud_negativa(parametros, serie):
    """Log-verosimilitud negativa de un GARCH(1,1) gaussiano. Ver Sesión 3, §3.4."""
    omega, alpha, beta = parametros
    sigma2 = np.zeros(len(serie))
    sigma2[0] = np.var(serie)
    for t in range(1, len(serie)):
        sigma2[t] = omega + alpha * serie[t - 1] ** 2 + beta * sigma2[t - 1]
    if np.any(sigma2 <= 1e-8):
        return 1e10
    return 0.5 * np.sum(np.log(2 * np.pi) + np.log(sigma2) + serie ** 2 / sigma2)


def correccion_arfima(residuos, d, h):
    """Pronóstico h pasos del componente ARFIMA(0,d,0) de los residuos.

    Diferencia fraccionalmente los residuos con d, pronostica cero (ruido blanco)
    y vuelve a integrar con -d. Lo que queda es la persistencia que la memoria
    larga añadiría al pronóstico del SARIMA (en escala logarítmica).
    """
    r = np.asarray(residuos, dtype=float)
    extendida = np.concatenate([aplicar_filtro_fraccional(r, d), np.zeros(h)])
    return aplicar_filtro_fraccional(extendida, -d)[len(r):]


def estimar_garch11(residuos, h):
    """Estima un GARCH(1,1) por MLE sobre los residuos estandarizados.

    Devuelve un diccionario con omega, alpha, beta, persistencia, convergencia,
    residuos estandarizados por sigma_t y el factor de escala de la desviación
    estándar condicional para los h meses siguientes (1 = igual a la varianza
    incondicional).
    """
    r = np.asarray(residuos, dtype=float)
    s = r.std()
    u = r / s
    res = minimize(garch_log_verosimilitud_negativa, x0=[0.05, 0.10, 0.80], args=(u,),
                   bounds=[(1e-6, 5.0), (1e-6, 0.999), (1e-6, 0.999)],
                   constraints=({"type": "ineq", "fun": lambda p: 0.999 - p[1] - p[2]},),
                   method="SLSQP")
    om, al, be = res.x
    sigma2 = np.zeros(len(u)); sigma2[0] = u.var()
    for t in range(1, len(u)):
        sigma2[t] = om + al * u[t - 1] ** 2 + be * sigma2[t - 1]
    futura = np.zeros(h)
    futura[0] = om + al * u[-1] ** 2 + be * sigma2[-1]
    for i in range(1, h):
        futura[i] = om + (al + be) * futura[i - 1]
    incondicional = om / (1 - al - be)
    return {"omega": om, "alpha": al, "beta": be, "persistencia": al + be,
            "convergencia": bool(res.success),
            "residuos_estandarizados": u / np.sqrt(sigma2),
            "escala_futura": np.sqrt(futura / incondicional)}


def metricas(real, pred):
    """RMSE y MAE en la escala de los datos (GWh)."""
    e = np.asarray(real, float) - np.asarray(pred, float)
    return {"RMSE": float(np.sqrt(np.mean(e ** 2))), "MAE": float(np.mean(np.abs(e)))}


def cobertura_y_ancho(real, inferior, superior):
    """Proporción de meses dentro del intervalo y ancho promedio del intervalo (GWh)."""
    real, inferior, superior = (np.asarray(x, float) for x in (real, inferior, superior))
    return {"Cobertura": float(np.mean((real >= inferior) & (real <= superior))),
            "Ancho medio": float(np.mean(superior - inferior))}

In [ ]:
df = pd.read_csv("consumo_electrico_litoral_depurado.csv", parse_dates=["fecha"], index_col="fecha")
df.index.freq = "MS"
y = df["consumo_gwh"]
log_y = np.log(y)

# Partición oficial: entrenamiento 2015-2022, validación 2023. 2024 no se utiliza esta semana.
H = 12
log_train, log_val = log_y[:"2022-12"], log_y["2023-01":"2023-12"]
y_val = y["2023-01":"2023-12"]
print(f"Entrenamiento: {len(log_train)} meses | Validación: {len(log_val)} meses")

In [ ]:
# ==============================================================================
# COMPLETE: el modelo SARIMA que seleccionó en la Sumativa 2 (misma especificación)
# ==============================================================================
ORDEN = None              # por ejemplo (p, d, q)
ORDEN_ESTACIONAL = None   # por ejemplo (P, D, Q, 12)
USAR_COVID = False        # True si incluyó la indicadora covid como variable exógena

assert ORDEN is not None and ORDEN_ESTACIONAL is not None, "Complete ORDEN y ORDEN_ESTACIONAL con su modelo de la Sumativa 2."

In [ ]:
# Reestimación del SARIMA sobre el entrenamiento (escala logarítmica)
X_train = df[["covid"]][:"2022-12"] if USAR_COVID else None
X_val = df[["covid"]].loc["2023-01":"2023-12"] if USAR_COVID else None

sarima = SARIMAX(log_train, exog=X_train, order=ORDEN, seasonal_order=ORDEN_ESTACIONAL).fit(disp=False)
inicio = ORDEN[1] + ORDEN_ESTACIONAL[1] * ORDEN_ESTACIONAL[3]      # se descartan las primeras d + D·s observaciones
residuos = sarima.resid.iloc[inicio:]

pron = sarima.get_forecast(H, exog=X_val)
media_log = pron.predicted_mean
ic_log = pron.conf_int(alpha=0.05)
sd_log = (ic_log.iloc[:, 1] - media_log) / 1.96

naive = y["2022-01":"2022-12"].to_numpy()                          # naive estacional
print(f"AIC = {sarima.aic:.1f} | residuos utilizables: {len(residuos)}")

## 1. Modelamiento de memoria larga (ARFIMA / diferenciación fraccional)

In [ ]:
# Su código aquí. Sugerencia de pasos:
#  - ACF de `residuos` hasta el rezago 36
#  - H = calcular_exponente_hurst(residuos); d = H - 0.5   (¿qué tan confiable es con ~80 observaciones?)
#  - pronóstico con corrección ARFIMA(0,d,0): media_log + correccion_arfima(residuos, d, H)

Interpretación (máx. 250 palabras).

## 2. Modelamiento de volatilidad (GARCH)

In [ ]:
# Su código aquí. Sugerencia de pasos:
#  - ACF de residuos**2; acorr_ljungbox(residuos**2, lags=[12]); het_arch(residuos, nlags=12)
#  - g = estimar_garch11(residuos, H): omega, alpha, beta, persistencia y diagnóstico de g["residuos_estandarizados"]
#  - bandas GARCH al 95 %: media_log ± 1.96 · sd_log · g["escala_futura"]  (llevar a GWh con np.exp)

Interpretación (máx. 250 palabras).

## 3. Contraste con el modelo clásico y consistencia de conclusiones

In [ ]:
# Su código aquí: tabla con metricas(y_val, ...) para el SARIMA, el SARIMA con corrección ARFIMA
# y el naive estacional, y cobertura_y_ancho(y_val, ...) para los intervalos del SARIMA y con bandas GARCH.

Interpretación (máx. 250 palabras).

## 4. Selección del modelo con evidencia cuantitativa

In [ ]:
# Su código aquí: pronóstico de validación 2023 del modelo elegido, con su intervalo al 95 %.

Interpretación (máx. 250 palabras). Este es el modelo que utilizará en la Semana 4.

## Conclusiones

## Referencias (APA)

## Declaración de uso de asistentes de IA

| Pregunta | Respuesta |
|---|---|
| ¿Utilizó asistentes de IA en esta entrega? | |
| ¿En qué partes específicas? | |
| ¿Qué verificó de forma independiente y cómo? | |